In [1]:
import sys, os
import mujoco
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import time
import cv2
from PIL import Image, ImageDraw
from IPython.display import display
from OpenGL.GL import *
import time
sys.path.append('../../package/kinematics_helper/')
sys.path.append('../../package/mujoco_helper/')
sys.path.append('../../package/utility/')
sys.path.append('../../package/openmanipulator/')

from ik import *
from mujoco_parser import *
from utils import *
from transforms import *
from ik_utils import *
import datetime
from dynamixel_sdk import *
from motor import *
from pos_utils import *
from ik_utils import interpolate_and_smooth_nd

print ("MuJoCo:[%s]"%(mujoco.__version__))
"""
requirements:
mujoco
imageio
matplotlib
dynamixel-sdk
"""


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/anaconda3/envs/openmanipulatorx/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/anaconda3/envs/openmanipulatorx/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/anaconda3/envs/openmanipulatorx/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/envs/openmanipulatorx/lib/python3.10/site-packages/traitlets/config/application.py",

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

# Prerequisite

In [2]:
"""codes to make glfw use GPU in rendering time"""

os.environ["__GLX_VENDOR_LIBRARY_NAME"] = "nvidia"
os.environ["__NV_PRIME_RENDER_OFFLOAD"] = "1"
os.environ["__VK_LAYER_NV_optimus"] = "NVIDIA_only"


In [3]:
"""canvas width, canvas height"""

canvas_width=0.125
canvas_height=0.179

In [4]:
def rpy_deg2r(r):
    r_rad=np.deg2rad(r)
    return rpy2r(r_rad)

# OpenManipulator-X

In [5]:
tutorial=False

In [6]:
if tutorial:
    openmanipulator_path = "../../asset/openmanipulator_x/scene_openmanipulator_x_rilab.xml"

    env=MuJoCoParserClass(name='OpenManipulator-X', rel_xml_path=openmanipulator_path,verbose=True)

## Forward Kinematics / Dynamics of OpenManipulator-X

In [7]:
if tutorial:
    env.open_interactive_viewer()

In [8]:
if tutorial:
    rev_and_pri_joint_idxs=np.where(np.isin(env.joint_types, 
                                            [mujoco.mjtJoint.mjJNT_HINGE, mujoco.mjtJoint.mjJNT_SLIDE]))[0].astype(np.int32)
    rev_and_pri_joint_names = [env.joint_names[joint_idx] for joint_idx in rev_and_pri_joint_idxs]
    rev_and_pri_joint_mins = env.joint_ranges[rev_and_pri_joint_idxs,0]
    rev_and_pri_joint_maxs = env.joint_ranges[rev_and_pri_joint_idxs,1]
    n_rev_and_pri_joints = len(rev_and_pri_joint_names)

### FK

In [9]:
if tutorial:
    # Initialize slider control
    env.reset(step=True)
    # init_qpos = env.get_qpos_joints(joint_names=env.rev_joint_names)
    init_qpos = env.get_qpos_joints(joint_names=rev_and_pri_joint_names)
    sliders = MultiSliderClass(
        n_slider      = n_rev_and_pri_joints,
        title         = 'Sliders for [%s] Control'%(env.name),
        window_width  = 600,
        window_height = 800,
        x_offset      = 50,
        y_offset      = 100,
        slider_width  = 350,
        label_texts   = rev_and_pri_joint_names,
        slider_mins   = rev_and_pri_joint_mins,
        slider_maxs   = rev_and_pri_joint_maxs,
        slider_vals   = init_qpos,
        verbose       = False,
    )
    idxs_fwd = env.get_idxs_fwd(joint_names=rev_and_pri_joint_names)
    # Loop
    env.init_viewer(transparent=True)
    while env.is_viewer_alive():
        # Update
        sliders.update()
        env.forward(q=sliders.get_slider_values(),joint_idxs=idxs_fwd)
        # Render
        if env.loop_every(tick_every=10):
            env.plot_joint_axis(axis_len=0.025,axis_r=0.005) # revolute joints
            env.plot_links_between_bodies(rgba=(0,0,0,1),r=0.001) # link information
            env.plot_contact_info()
            env.render()
    # Close slider
    sliders.close()
    print ("Done.")

### FD

In [10]:
if tutorial:
    init_ctrl = env.get_ctrl(ctrl_names=env.ctrl_names)
    sliders = MultiSliderClass(
        n_slider      = env.n_ctrl,
        title         = 'Sliders for [%s] Control'%(env.name),
        window_width  = 600,
        window_height = 350,
        x_offset      = 50,
        y_offset      = 100,
        slider_width  = 300,
        label_texts   = env.ctrl_names,
        slider_mins   = env.ctrl_mins,
        slider_maxs   = env.ctrl_maxs,
        slider_vals   = init_ctrl,
        verbose       = False,
    )
    # Arrange objects
    env.reset(step=True)
    obj_names = env.get_body_names(prefix='obj_')
    n_obj = len(obj_names)
    obj_xyzs = sample_xyzs(
        n_sample  = n_obj,
        x_range   = [+0.6,+1.0],
        y_range   = [-0.45,+0.45],
        z_range   = [0.8,0.81],
        min_dist  = 0.2,
        xy_margin = 0.0
    )
    for obj_idx in range(n_obj):
        env.set_p_base_body(body_name=obj_names[obj_idx],p=obj_xyzs[obj_idx,:])
        env.set_R_base_body(body_name=obj_names[obj_idx],R=np.eye(3,3))
    env.set_geom_color(body_names_to_color=obj_names,rgba_list=get_colors(n_obj))
    # Loop
    env.init_viewer(transparent=True)
    while env.is_viewer_alive():
        # Update
        sliders.update()
        env.step(ctrl=sliders.get_slider_values())
        # Render
        if env.loop_every(tick_every=50):
            env.plot_time()
            env.plot_joint_axis(axis_len=0.025,axis_r=0.005) # revolute joints
            env.plot_links_between_bodies(rgba=(0,0,0,1),r=0.001) # link information
            env.plot_contact_info()
            env.render()
    # Close slider
    sliders.close()
    print ("Done.")

## Inverse Kinematics of OpenManipulator-X

In [11]:
if tutorial:
    sliders = MultiSliderClass( # Slider for EE control
        n_slider      = 7,
        title         = 'Sliders for [%s] Control'%(env.name),
        window_width  = 450,
        window_height = 300,
        x_offset      = 0,
        y_offset      = 100,
        slider_width  = 300,
        label_texts   = ['X','Y','Z','Roll-deg','Pitch-deg','Yaw-deg','Gripper'],
        slider_mins   = [-1,-1,-1,-180,-180,-180,0],
        slider_maxs   = [+1,+1,+1,+180,+180,+180,2],
        slider_vals   = [0,0,0,0,0,0,0],
        resolutions   = [0.02,0.02,0.02,3.6,3.6,3.6,0.04], # range/50
        verbose       = False,
    )
    joint_names = ["joint1","joint2","joint3","joint4","gripper_left_joint","gripper_right_joint"]
    q0 = np.deg2rad([0,0,0,0,0,0])
    p0 = env.get_p_body(body_name='world')
    R0 = rpy_deg2r([0,0,0])
    env.init_viewer(
        title       = 'Tabletop',
        transparent = False,
        azimuth     = 133,
        distance    = 1.2,
        elevation   = -42.4,
        lookat      = (0.06,-0.07,0.31),
    )
    env.reset() # reset
    env.forward(q=q0,joint_names=joint_names) # initial position

    # Move object positions
    obj_names = env.get_body_names(prefix='obj_')
    n_obj = len(obj_names)
    obj_xyzs = sample_xyzs(
        n_sample  = n_obj,
        x_range   = [+0.6,+1.0],
        y_range   = [-0.45,+0.45],
        z_range   = [0.8,0.81],
        min_dist  = 0.2,
        xy_margin = 0.0
    )
    for obj_idx in range(n_obj):
        env.set_p_base_body(body_name=obj_names[obj_idx],p=obj_xyzs[obj_idx,:])
        env.set_R_base_body(body_name=obj_names[obj_idx],R=np.eye(3,3))
    env.set_geom_color(body_names_to_color=obj_names,rgba_list=get_colors(n_obj))
        
    # Loop
    q_ik_init = q0.copy()
    while env.is_viewer_alive():
        
        # Update
        sliders.update() # update slider
        xyzrpyg = sliders.get_slider_values()
        qpos,ik_err_stack,ik_info = solve_ik(
            env                = env,
            joint_names_for_ik = joint_names,
            body_name_trgt     = 'end_effector_target',
            q_init             = q_ik_init,
            p_trgt             = xyzrpyg[:3]+p0,
            R_trgt             = rpy_deg2r(xyzrpyg[3:6])@R0,
            max_ik_tick        = 500,
            ik_stepsize        = 1.0,
            ik_eps             = 1e-2,
            ik_th              = np.radians(5.0),
            render             = False,
            verbose_warning    = False,
        )
        ik_err = np.abs(ik_err_stack).max() # IK error
        if ik_err < 1e-2: q_ik_init = qpos.copy()
        else: q_ik_init = q0.copy()    
        env.forward(q=qpos,joint_names=joint_names) # kinematic update

        # Click handler
        xyz_click,flag_click = env.get_xyz_left_double_click()
        if flag_click: print ("[CLICKED] p:%s"%(xyz_click))
        
        # Render 
        if env.loop_every(tick_every=10):
            env.plot_T(
                T=env.get_T_body(body_name='world'),
                axis_len=0.5,print_xyz=False)
            env.plot_text(
                p=env.get_p_body(body_name='world')+np.array([0,0,0.5]),
                label = 'time:[%.2f]sec ik_err:[%.3f]'%(env.get_sim_time(),ik_err))
            env.plot_body_T(body_name='end_effector_target',axis_len=0.1,axis_width=0.005)
            env.plot_contact_info(
                r_arrow=0.005,h_arrow=0.1,rgba_contact=(1,0,0,0.5),plot_sphere=False)
            plot_ik_info(env=env,ik_info=ik_info)
            if xyz_click is not None:
                env.plot_sphere(p=xyz_click,r=0.01,rgba=(1,0,0,0.5))
            env.render()
        if env.loop_every(tick_every=5000): 
            img = env.grab_image()
            plt.figure(figsize=(6,4)); plt.imshow(img); 
            plt.title('tick:[%d] time:[%.2f]sec'%
                    (env.tick,env.get_sim_time()),fontsize=9)
            plt.axis('off'); plt.show()

    # Close
    env.close_viewer()
    sliders.close()
    print ("Done.")